docker-compose up --build -d

RUBY COM CACHE

for U in 1 5 10 25 50; do
  echo "Testando com $U usuários..."
  locust -f locustfile.py \
    --host http://localhost:4567 \
    --users $U \
    --spawn-rate 2 \
    --run-time 60s \
    --headless \
    --csv resultados/ruby_cache_${U}u
  echo "Feito!"
done

docker-compose down

RUBY SEM CACHE

docker-compose -f docker-compose-ruby-no-cache.yml up --build -d

for U in 1 5 10 25 50; do
  locust -f locustfile.py \
    --host http://localhost:4567 \
    --users $U --spawn-rate 2 --run-time 60s --headless \
    --csv resultados/ruby_nocache_${U}u
done

docker-compose -f docker-compose-ruby-no-cache.yml down

PYTHON COM CACHE

docker-compose -f docker-compose-python.yml up --build -d

for U in 1 5 10 25 50; do
  locust -f locustfile.py \
    --host http://localhost:5000 \
    --users $U --spawn-rate 2 --run-time 60s --headless \
    --csv resultados/python_cache_${U}u
done

docker-compose -f docker-compose-python.yml down

PYTHON SEM CACHE

docker-compose -f docker-compose-python-no-cache.yml up --build -d

for U in 1 5 10 25 50; do
  locust -f locustfile.py \
    --host http://localhost:5000 \
    --users $U --spawn-rate 2 --run-time 60s --headless \
    --csv resultados/python_nocache_${U}u
done

docker-compose -f docker-compose-python-no-cache.yml down

In [ ]:
# Instalar as bibliotecas de análise (se ainda não tiver)
!pip install pandas matplotlib numpy --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# Configurações visuais dos gráficos
%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,          # qualidade da imagem
    'font.size': 11,
    'axes.grid': True,          # linhas de grade
    'grid.alpha': 0.4,
    'axes.spines.top': False,   # remove borda superior
    'axes.spines.right': False, # remove borda direita
})

print('Bibliotecas importadas com sucesso!')
print('Versão pandas:', pd.__version__)
print('Versão numpy:', np.__version__)


In [ ]:
def carregar_resultados_reais(pasta='resultados'):
    """
    Lê os CSVs gerados pelo Locust e monta um DataFrame.
    Formato esperado do nome do arquivo:
        ruby_cache_10u_stats.csv
        python_nocache_25u_stats.csv
        etc.
    """
    import glob

    mapa_nomes = {
        'ruby_cache':       'Ruby + Cache',
        'ruby_nocache':     'Ruby + Sem Cache',
        'python_cache':     'Python + Cache',
        'python_nocache':   'Python + Sem Cache',
    }

    arquivos = sorted(glob.glob(os.path.join(pasta, '*_stats.csv')))

    if not arquivos:
        print(f'Nenhum arquivo _stats.csv encontrado em {pasta}/')
        print('Rode os testes primeiro (Módulo 7) e volte aqui.')
        return None

    linhas = []
    for arq in arquivos:
        nome = os.path.basename(arq).replace('_stats.csv', '')  # ex: ruby_cache_10u
        partes = nome.rsplit('_', 1)                             # ['ruby_cache', '10u']
        prefixo = partes[0]
        usuarios = int(partes[1].replace('u', '')) if len(partes) == 2 else 0
        cenario = mapa_nomes.get(prefixo, prefixo)

        df_csv = pd.read_csv(arq)
        # Pega só a linha de totais
        agg = df_csv[df_csv['Name'] == 'Aggregated'].iloc[0]

        linhas.append({
            'cenario':    cenario,
            'usuarios':   usuarios,
            'mediana_ms': agg.get('50%',   agg.get('Median Response Time', 0)),
            'p95_ms':     agg.get('95%',   0),
            'p99_ms':     agg.get('99%',   0),
            'media_ms':   agg.get('Average Response Time', 0),
            'rps':        agg.get('Requests/s', 0),
            'falhas':     agg.get('Failure Count', 0),
        })

    df_real = pd.DataFrame(linhas).sort_values(['cenario', 'usuarios'])
    print(f'{len(df_real)} testes carregados!')
    return df_real


# Tenta carregar resultados reais
import glob
csvs_existentes = glob.glob('resultados/*_stats.csv')

if csvs_existentes:
    print(f'Encontrados {len(csvs_existentes)} arquivo(s) de resultado!')
    df_real = carregar_resultados_reais()
    if df_real is not None:
        print()
        print(df_real.to_string(index=False))
else:
    print('Ainda sem resultados reais — usando dados simulados nas análises acima.')
    print()
    print('Quando tiver resultados:')
    print('  1. Execute os comandos do Módulo 7 no terminal')
    print('  2. Volte aqui e execute: df_real = carregar_resultados_reais()')
    print('  3. Use df_real nos gráficos no lugar de medianas/p95/rps')


---
# Gráficos com Resultados Reais

Execute as células abaixo **depois** de `df_real` estar carregado acima.

In [ ]:
import glob

# Recarrega resultados (útil se novos CSVs foram gerados)
df_real = carregar_resultados_reais()

if df_real is not None:
    print(df_real.to_string(index=False))


In [ ]:
if df_real is None:
    raise SystemExit("Sem dados. Execute os testes primeiro.")

os.makedirs('resultados', exist_ok=True)

CORES = {
    'Ruby + Cache':       '#1565C0',
    'Ruby + Sem Cache':   '#E53935',
    'Python + Cache':     '#2E7D32',
    'Python + Sem Cache': '#F57F17',
}
MARCADORES = {
    'Ruby + Cache': 'o',
    'Ruby + Sem Cache': 's',
    'Python + Cache': '^',
    'Python + Sem Cache': 'D',
}

cargas   = sorted(df_real['usuarios'].unique())
cenarios = sorted(df_real['cenario'].unique())
print('Cenários disponíveis:', cenarios)
print('Cargas disponíveis:  ', cargas)


## Gráfico 1 — Mediana do Tempo de Resposta vs Usuários

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for cenario in cenarios:
    sub = df_real[df_real['cenario'] == cenario].sort_values('usuarios')
    ax.plot(
        sub['usuarios'], sub['mediana_ms'],
        f"-{MARCADORES.get(cenario, 'o')}",
        label=cenario,
        color=CORES.get(cenario, '#333'),
        linewidth=2.2, markersize=7,
    )
    for _, row in sub.iterrows():
        ax.annotate(f"{int(row['mediana_ms'])}ms",
                    (row['usuarios'], row['mediana_ms']),
                    textcoords='offset points', xytext=(0, 8),
                    ha='center', fontsize=7.5,
                    color=CORES.get(cenario, '#333'))

ax.axhline(1000, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='1 segundo')
ax.set_title('Mediana do Tempo de Resposta vs Número de Usuários', fontsize=13, fontweight='bold')
ax.set_xlabel('Usuários Virtuais Simultâneos')
ax.set_ylabel('Mediana (ms)')
ax.set_xticks(cargas)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('resultados/real_grafico1_mediana.png', bbox_inches='tight')
plt.show()


## Gráfico 2 — Percentil 95 vs Usuários

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for cenario in cenarios:
    sub = df_real[df_real['cenario'] == cenario].sort_values('usuarios')
    ax.plot(
        sub['usuarios'], sub['p95_ms'],
        f"-{MARCADORES.get(cenario, 'o')}",
        label=cenario,
        color=CORES.get(cenario, '#333'),
        linewidth=2.2, markersize=7,
    )

ax.axhline(1000, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='1 segundo')
ax.set_title('Percentil 95 do Tempo de Resposta vs Número de Usuários', fontsize=13, fontweight='bold')
ax.set_xlabel('Usuários Virtuais Simultâneos')
ax.set_ylabel('P95 (ms) — 95% das respostas abaixo deste valor')
ax.set_xticks(cargas)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('resultados/real_grafico2_p95.png', bbox_inches='tight')
plt.show()


## Gráfico 3 — Throughput (Requisições por Segundo)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for cenario in cenarios:
    sub = df_real[df_real['cenario'] == cenario].sort_values('usuarios')
    ax.plot(
        sub['usuarios'], sub['rps'],
        f"-{MARCADORES.get(cenario, 'o')}",
        label=cenario,
        color=CORES.get(cenario, '#333'),
        linewidth=2.2, markersize=7,
    )
    for _, row in sub.iterrows():
        ax.annotate(f"{row['rps']:.0f}",
                    (row['usuarios'], row['rps']),
                    textcoords='offset points', xytext=(0, 8),
                    ha='center', fontsize=7.5,
                    color=CORES.get(cenario, '#333'))

ax.set_title('Throughput (RPS) vs Número de Usuários', fontsize=13, fontweight='bold')
ax.set_xlabel('Usuários Virtuais Simultâneos')
ax.set_ylabel('Requisições por Segundo (RPS)')
ax.set_xticks(cargas)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('resultados/real_grafico3_rps.png', bbox_inches='tight')
plt.show()


## Gráfico 4 — P50 / P95 / P99 por Cenário (carga máxima)

In [ ]:
carga_max = max(cargas)
df_max = df_real[df_real['usuarios'] == carga_max]

x = np.arange(len(cenarios))
largura = 0.25

fig, ax = plt.subplots(figsize=(12, 6))

p50_vals = [df_max[df_max['cenario'] == c]['mediana_ms'].values[0]
            if len(df_max[df_max['cenario'] == c]) else 0 for c in cenarios]
p95_vals = [df_max[df_max['cenario'] == c]['p95_ms'].values[0]
            if len(df_max[df_max['cenario'] == c]) else 0 for c in cenarios]
p99_vals = [df_max[df_max['cenario'] == c]['p99_ms'].values[0]
            if len(df_max[df_max['cenario'] == c]) else 0 for c in cenarios]

b50 = ax.bar(x - largura, p50_vals, largura, label='P50 (mediana)', color='#42A5F5', edgecolor='white')
b95 = ax.bar(x,           p95_vals, largura, label='P95',           color='#FFA726', edgecolor='white')
b99 = ax.bar(x + largura, p99_vals, largura, label='P99',           color='#EF5350', edgecolor='white')

for bars in [b50, b95, b99]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 20,
                    f'{int(h)}ms', ha='center', va='bottom', fontsize=8)

ax.set_title(f'Distribuição de Percentis por Cenário ({carga_max} usuários)', fontsize=13, fontweight='bold')
ax.set_ylabel('Tempo de Resposta (ms)')
ax.set_xticks(x)
ax.set_xticklabels(cenarios, rotation=10, ha='right')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('resultados/real_grafico4_percentis.png', bbox_inches='tight')
plt.show()


## Tabela Resumo Final

In [ ]:
df_resumo = (
    df_real[df_real['usuarios'] == carga_max]
    .set_index('cenario')[['mediana_ms', 'p95_ms', 'p99_ms', 'rps', 'falhas']]
    .rename(columns={
        'mediana_ms': 'Mediana (ms)',
        'p95_ms':     'P95 (ms)',
        'p99_ms':     'P99 (ms)',
        'rps':        'RPS',
        'falhas':     'Falhas',
    })
)

pior = df_resumo['Mediana (ms)'].max()
df_resumo['Speedup'] = (pior / df_resumo['Mediana (ms)']).round(1).astype(str) + 'x'

print('=' * 70)
print(f'  RESUMO — {carga_max} Usuários Virtuais Simultâneos')
print('=' * 70)
print(df_resumo.to_string())
print()
print('Gráficos salvos:')
for f in sorted(glob.glob('resultados/real_grafico*.png')):
    print(f'  {f}')
